In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
BASE = Path("..")

UPSET_DIR = BASE / "upset_frequency"
OUT_DIR   = BASE / "skill_luck_decomposition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEAGUE_MAP = {
    "Bundesliga": "Bundesliga",
    "La Liga": "La Liga",
    "Premier League": "Premier League",
    "Serie A": "Serie A",
    "bundesliga": "Bundesliga",
    "la_liga": "La Liga",
    "premier_league": "Premier League",
    "serie_a": "Serie A",
}

def canon_league(name: str) -> str:
    return LEAGUE_MAP.get(str(name).strip(), str(name).strip())

In [3]:
# Helper to compute q for vectors (pandas Series)

# Using the formula: actual = q*skill + (1-q)*luck  =>  q = (actual - luck)/(skill - luck)

def compute_q_series(actual, skill, luck):
    denom = skill - luck
    q = (actual - luck) / denom

    # replace the value if denominator = 0 and values tending to infinity
    q = q.replace([np.inf, -np.inf], np.nan)
    
    return q.clip(0, 1)

In [10]:
def read_upset_summary(subfolder: str, filename: str) -> pd.DataFrame:
    path = UPSET_DIR / subfolder / filename
    print(f"Reading: {path}")
    df = pd.read_csv(path)

    # ---- Case 1: already long format with league + season ----
    if "league" in df.columns and "season" in df.columns:
        # Try to find which column holds the upset frequency
        freq_col = None
        for cand in ["upset_frequency_avg", "upset_frequency", "upset_frequency_seed_1"]:
            if cand in df.columns:
                freq_col = cand
                break

        if freq_col is None:
            raise ValueError(
                f"No upset-frequency column found in {path}. "
                f"Columns: {list(df.columns)}"
            )

        long = df[["league", "season", freq_col]].copy()
        long = long.rename(columns={freq_col: "upset_frequency"})
        long["league"] = long["league"].apply(canon_league)
        long["season"] = long["season"].astype(int)
        return long

    # ---- Case 2: wide format: one column per league (pure_luck_*) ----
    league_cols = [c for c in df.columns if c in LEAGUE_MAP.keys()]
    if not league_cols:
        raise ValueError(
            f"No league columns found in {path}. Columns: {list(df.columns)}"
        )

    # Assume remaining column is 'season'
    id_cols = [c for c in df.columns if c not in league_cols]
    if len(id_cols) != 1:
        raise ValueError(
            f"Expected exactly one id column (season) in {path}, got {id_cols}"
        )
    season_col = id_cols[0]

    long = df.melt(
        id_vars=season_col,
        value_vars=league_cols,
        var_name="league_raw",
        value_name="upset_frequency",
    )

    long["league"] = long["league_raw"].apply(canon_league)
    long["season"] = long[season_col].astype(int)

    return long[["league", "season", "upset_frequency"]]


In [11]:
def build_upset_frequency_skill_luck():
    """
    creates a wide csv with upset-frequency skill–luck decomposition.

    output columns:
      league, season,
      upset_frequency_actual,
      upset_frequency_skill,
      upset_frequency_luck_results_avg,
      upset_frequency_luck_goals_avg,
      q_results,
      q_goals
    """

    # read every input
    df_actual = read_upset_summary(
        "actual", "season_upset_frequency_summary.csv"
    )
    df_skill = read_upset_summary(
        "pure_skill", "season_upset_frequency_summary.csv"
    )
    df_luck_goals = read_upset_summary(
        "pure_luck_goals_based", "season_upset_frequency_summary_all_seeds.csv"
    )
    df_luck_results = read_upset_summary(
        "pure_luck_result_based", "season_upset_frequency_summary_all_seeds.csv"
    )

    key = ["league", "season"]

    # merge into one table
    wide = (
        df_actual.rename(columns={"upset_frequency": "upset_frequency_actual"})[key + ["upset_frequency_actual"]]
        .merge(
            df_skill.rename(columns={"upset_frequency": "upset_frequency_skill"})[key + ["upset_frequency_skill"]],
            on=key,
            how="inner",
        )
        .merge(
            df_luck_results.rename(
                columns={"upset_frequency": "upset_frequency_luck_results_avg"}
            )[key + ["upset_frequency_luck_results_avg"]],
            on=key,
            how="inner",
        )
        .merge(
            df_luck_goals.rename(
                columns={"upset_frequency": "upset_frequency_luck_goals_avg"}
            )[key + ["upset_frequency_luck_goals_avg"]],
            on=key,
            how="inner",
        )
    )

    # compute the q-values
    wide["q_results"] = compute_q_series(
        wide["upset_frequency_actual"],
        wide["upset_frequency_skill"],
        wide["upset_frequency_luck_results_avg"],
    )

    wide["q_goals"] = compute_q_series(
        wide["upset_frequency_actual"],
        wide["upset_frequency_skill"],
        wide["upset_frequency_luck_goals_avg"],
    )

    # order the columns nicely
    wide = wide[
        [
            "league",
            "season",
            "upset_frequency_actual",
            "upset_frequency_skill",
            "upset_frequency_luck_results_avg",
            "upset_frequency_luck_goals_avg",
            "q_results",
            "q_goals",
        ]
    ].sort_values(["league", "season"])

    # save csv
    out_path = OUT_DIR / "upset_frequency_skill_luck_decomposition.csv"
    wide.to_csv(out_path, index=False)
    print(f"Saved upset-frequency skill–luck decomposition to: {out_path}")

    return wide

In [12]:
df_upset_wide = build_upset_frequency_skill_luck()
df_upset_wide.head()

Reading: ../upset_frequency/actual/season_upset_frequency_summary.csv
Reading: ../upset_frequency/pure_skill/season_upset_frequency_summary.csv
Reading: ../upset_frequency/pure_luck_goals_based/season_upset_frequency_summary_all_seeds.csv
Reading: ../upset_frequency/pure_luck_result_based/season_upset_frequency_summary_all_seeds.csv
Saved upset-frequency skill–luck decomposition to: ../skill_luck_decomposition/upset_frequency_skill_luck_decomposition.csv


,league,season,upset_frequency_actual,upset_frequency_skill,upset_frequency_luck_results_avg,upset_frequency_luck_goals_avg,q_results,q_goals
0,Bundesliga,2004,0.357843,0.0,0.412908,0.381046,0.133360,0.060892
1,Bundesliga,2005,0.362745,0.0,0.408660,0.392157,0.112355,0.075000
2,Bundesliga,2006,0.380719,0.0,0.414216,0.405556,0.080868,0.061241
3,Bundesliga,2007,0.356209,0.0,0.433824,0.392810,0.178908,0.093178
4,Bundesliga,2008,0.326797,0.0,0.413235,0.389542,0.209174,0.161074
